In [ ]:
import pickle
import gzip

import utils

import loompy
import pandas as pd
import numpy as np
import scipy.sparse as sp

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
%%time
data = pd.read_table("data/GSE63472_P14Retina_merged_digital_expression.txt.gz", index_col=0)
data.head()

In [ ]:
%%time
cluster_ids = pd.read_table("data/retina_clusteridentities.txt", header=None, index_col=0, squeeze=True)

In [ ]:
cluster_ids.head()

In [5]:
# Reorder
cluster_ids = cluster_ids[data.columns.values]

In [ ]:
data.shape

In [7]:
# Only use cells where metadata is available
ind = data.columns.isin(cluster_ids.index)
data = data.loc[:, ind]

In [ ]:
data.shape, cluster_ids.shape

In [9]:
mask = ~cluster_ids.isna()
data = data.loc[:, mask.values]
cluster_ids = cluster_ids[mask]

In [10]:
assert not cluster_ids.isna().any(), "Did not properly remove cells with NaN label"

In [ ]:
data.shape, cluster_ids.shape

In [ ]:
%%time
counts = sp.csr_matrix(data.values)
counts

In [ ]:
%%time
cpm_counts = utils.calculate_cpm(counts, axis=0)
log_counts = utils.log_normalize(cpm_counts)

In [ ]:
cell_types = cluster_ids.astype(object)

cell_types.loc[cell_types == 1] = "Horizontal cells"
cell_types.loc[cell_types == 2] = "Retinal ganglion cells"
cell_types.loc[cell_types.isin(range(3, 24))] = "Amacrine cells"
cell_types.loc[cell_types == 24] = "Rods"
cell_types.loc[cell_types == 25] = "Cones"
cell_types.loc[cell_types.isin(range(26, 34))] = "Bipolar cells"
cell_types.loc[cell_types == 34] = "Muller glia"
cell_types.loc[cell_types == 35] = "Astrocytes"
cell_types.loc[cell_types == 36] = "Fibroblasts"
cell_types.loc[cell_types == 37] = "Vascular endothelium"
cell_types.loc[cell_types == 38] = "Pericytes"
cell_types.loc[cell_types == 39] = "Microglia"

cell_types.value_counts()

## Preprocess data set

### Dropout based feature selection

In [ ]:
%time gene_mask = utils.select_genes(counts.T, n=3000, threshold=0)

In [16]:
x = log_counts.T[:, gene_mask].toarray()
x.shape

(44808, 3000)

### Standardize data

In [17]:
x -= x.mean(axis=0)
x /= x.std(axis=0)

### PCA preprocessing

In [ ]:
%%time
U, S, V = np.linalg.svd(x, full_matrices=False)
U[:, np.sum(V, axis=1) < 0] *= -1
x_reduced = np.dot(U, np.diag(S))
x_reduced = x_reduced[:, np.argsort(S)[::-1]][:, :50]

In [ ]:
x_reduced.shape

In [ ]:
cell_types.shape

## Write data

In [21]:
data_dict = {"pca_50": x_reduced,
             "CellType1": cell_types.values.astype(str),
             "CellType2": cluster_ids.values.astype(str)}

In [ ]:
%%time
with gzip.open("data/macosko_2015.pkl.gz", "wb") as f:
    pickle.dump(data_dict, f)